<a href="https://colab.research.google.com/github/GuFerreiraV/notebooks-google-colab/blob/main/Notebook_An%C3%A1lise_de_Emo%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparando ambiente
Instalando bibliotecas, modelos e configurando o google drive para persistência de imagens e vídeos e gerando arquivo md com instruções para modelo generativo.

### Instalação de dependências

In [ ]:
!pip install ultralytics
!pip install deepface
!pip install langchain-ollama
!pip install --upgrade ultralytics
!pip install fpdf

In [ ]:
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 79 not upgraded.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRele

### Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
caminho_projeto = '/content/drive/MyDrive/notebookcolab/images'

if not os.path.exists(caminho_projeto):
    os.makedirs(caminho_projeto, exist_ok=True)
    print(f"Pasta criada com sucesso em: {caminho_projeto}")
else:
    print(f"A pasta já existe em: {caminho_projeto}")

A pasta já existe em: /content/drive/MyDrive/notebookcolab/images


### Arquivo de instrução

In [ ]:
conteudo_md = """### INSTRUÇÕES

# Persona e Contexto Operacional
Você é um **Módulo Analítico de Fusão Multimodal**. Sua função é receber dados estruturados de Visão Computacional (objetos e expressões faciais) e combiná-los com legendas de texto (quando fornecidas) para gerar uma síntese contextual da cena.

## Diretrizes de Comportamento
1. **Tom de Voz**: Estritamente técnico, objetivo e conciso.
2. **Escopo de Domínio**: Limite-se a analisar sentimentos, interações entre objetos e o contexto da cena.
3. **Restrição de Criatividade**: Proibido inventar elementos não presentes no JSON de entrada. Se a legenda diz "praia" mas o YOLO não detectou areia ou mar, priorize os dados visuais confirmados. Se não houver legenda, baseie a análise puramente nas evidências visuais confirmadas.
4. **Vocabulário Sugerido**: Utilize termos como "correlação", "inferência", "predomínio emocional", "co-ocorrência de objetos".
5. **Expansão Analítica**: Para cada item da 'SÍNTESE SEMÂNTICA', descreva a correlação técnica entre os objetos (YOLO) e a emoção (DeepFace). Se houver dissonância textual, justifique-a com base nas evidências visuais.

## Protocolo de Resolução de Conflitos
Em casos de discrepância entre os Dados Visuais (CV) e a Legenda Original, siga estas regras:
1. **Prioridade de Detecção**: Considere as detecções do YOLO e DeepFace como fatos observados (evidência física).
2. **Contextualização Textual**: Considere a legenda como a intenção ou o sentimento subjetivo do autor.
3. **Relato de Discrepância**: Se houver uma contradição clara (ex: Legenda feliz vs. Rosto triste), o modelo DEVE apontar a "Dissonância Semântica".
4. **Ausência de Legenda**: Caso não seja fornecida uma legenda, avalie a coerência da cena isoladamente (emoção versus ambiente/objetos)".

## Estrutura de Saída Obrigatória
Toda resposta deve seguir estritamente o formato de lista abaixo. Não inclua saudações, introduções ou conclusões.
- **DETECÇÃO VISUAL**: [Listar objetos relevantes detectados pelo YOLO]
- **ANÁLISE BIOMÉTRICA**: [Listar emoções predominantes detectadas pelo DeepFace]
- **CONTEXTO TEXTUAL**: [Resumo da intenção da legenda fornecida, ou "Legenda não fornecida"]
- **SÍNTESE SEMÂNTICA**: [Conclusão técnica sobre o cenário construído pela imagem e o texto]

## Exemplos de Referência (Few-Shot)

### EXEMPLO 1: CONCORDÂNCIA SEMÂNTICA
- **ENTRADA DE CV**: Objetos: [cachorro, grama, bola]. Emoções: [felicidade].
- **LEGENDA**: "Dia de brincar no parque!"
- **SAÍDA**:
    - **DETECÇÃO VISUAL**: Cachorro, grama e bola de brinquedo.
    - **ANÁLISE BIOMÉTRICA**: Expressão facial de felicidade confirmada.
    - **CONTEXTO TEXTUAL**: Atividade de lazer e celebração com animal de estimação.
    - **SÍNTESE SEMÂNTICA**: Concordância total. O ambiente visual de lazer reforça a mensagem positiva da legenda.
    - **OBSERVAÇÕES DE DISSONÂNCIA**: Nenhuma.

### EXEMPLO 2: DISSONÂNCIA SEMÂNTICA (O CASO DO CONTRASTE)
- **ENTRADA DE CV**: Objetos: [10 xícaras de café, 1 laptop]. Emoções: [tristeza, exaustão].
- **LEGENDA**: "Dia super produtivo e animado!"
- **SAÍDA**:
    - **DETECÇÃO VISUAL**: Laptop e alta densidade de recipientes de café.
    - **ANÁLISE BIOMÉTRICA**: Expressão facial de exaustão e tristeza.
    - **CONTEXTO TEXTUAL**: Afirmação de alta produtividade e ânimo.
    - **SÍNTESE SEMÂNTICA**: Dissonância crítica. Os indicadores biológicos sugerem fadiga extrema, contradizendo o termo "animado".
    - **OBSERVAÇÕES DE DISSONÂNCIA**: Contradição severa entre fadiga detectada e ânimo relatado. Nota: 9/10.

### EXEMPLO 3: ANÁLISE SEM LEGENDA (CENA PURAMENTE VISUAL)
- **ENTRADA DE CV**: Objetos: [1 laptop, 1 cadeira]. Emoções: [foco, neutralidade].
- **LEGENDA**: "Nenhuma legenda fornecida para esta imagem."
- **SAÍDA**:
    - **DETECÇÃO VISUAL**: Laptop e cadeira, sugerindo ambiente de trabalho ou estudo.
    - **ANÁLISE BIOMÉTRICA**: Expressão neutra indicando concentração.
    - **CONTEXTO TEXTUAL**: Legenda não fornecida.
    - **SÍNTESE SEMÂNTICA**: A análise puramente visual aponta para um cenário de foco produtivo. Há forte correlação entre os equipamentos detectados e a emoção biométrica rastreada.
    - **OBSERVAÇÕES DE DISSONÂNCIA**: Não aplicável (Contexto textual ausente).
"""

# Define o caminho onde o script orquestrador vai buscar
caminho_instrucoes = '/content/instructions.md'

with open(caminho_instrucoes, 'w', encoding='utf-8') as f:
    f.write(conteudo_md)

print(f"Arquivo {caminho_instrucoes} criado com sucesso!")

Arquivo /content/instructions.md criado com sucesso!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Ollama

In [ ]:
!ollama serve

Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [ ]:
!ollama pull gemma4:e2b

In [ ]:
import subprocess
import time

# Tenta encerrar instâncias anteriores para evitar conflitos
!pkill ollama

with open("ollama.log", "w") as log_file:
    process = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

print("⏳ Aguardando o servidor Ollama iniciar...")
time.sleep(10)

!curl -s http://localhost:11434/api/tags > /dev/null && print("Servidor Ollama online!") || print("Falha ao iniciar o servidor.")

⏳ Aguardando o servidor Ollama iniciar...
/bin/bash: -c: line 1: syntax error near unexpected token `"Servidor Ollama online!"'
/bin/bash: -c: line 1: `curl -s http://localhost:11434/api/tags > /dev/null && print("Servidor Ollama online!") || print("Falha ao iniciar o servidor.")'


# Desenvolvimento

1. **Camada de Visão (YOLOv8):** Responsável por identificar objetos físicos (pessoas, carros, acessórios). Se a confiança for baixa, o sistema aplica uma Redução Dinâmica para tentar capturar contexto relevante sem gerar falsos positivos.

2. **Camada Biométrica (DeepFace):** Focada em analisar as faces detectadas utilizando o backend retinaface. O sistema calcula a média emocional do grupo para definir o 'clima' da cena.

3. **Camada de Fusão (LLM):** O script orquestrador une as detecções visuais com a legenda do usuário e submete ao modelo Gemma:2b através do LangChain, buscando identificar concordâncias ou dissonâncias semânticas.

## YOLOv8

In [ ]:
from ultralytics import YOLO

model_yolo_nano = YOLO('yolov8n.pt')

results = model_yolo_nano.predict(source=caminho_projeto, conf=0.25)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

image 1/4 /content/drive/MyDrive/notebookcolab/images/people_neutral.jpg: 448x640 1 person, 340.7ms
image 2/4 /content/drive/MyDrive/notebookcolab/images/people_runs.jpg: 448x640 2 persons, 48.9ms
image 3/4 /content/drive/MyDrive/notebookcolab/images/peoples.jpg: 480x640 10 persons, 243.7ms
image 4/4 /content/drive/MyDrive/notebookcolab/images/rage people.jpg: 384x640 1 person, 1 cup, 1 chair, 1 potted plant, 149.7ms
Speed: 13.6ms preprocess, 195.7ms inference, 40.5ms postprocess per image at shape (1, 3, 384, 640)


### Contagem de objetos

In [ ]:
from collections import Counter

class ObjectDetected:
    def __init__(self, name, confidence):
        self.name = name
        self.confidence = confidence

# Função para extrair os dados brutos do YOLO e converter para sua estrutura
def formatar_objetos_yolo(resultados_yolo):
    # Filtrar por confiança (limite de 50%)
    objetos_filtrados = [
        obj.name for obj in resultados_yolo
        if obj.confidence >= 0.5
    ]

    # Se a lista estiver vazia, aplicar Redução Dinâmica (30%)
    if not objetos_filtrados:
        objetos_filtrados = [
            obj.name for obj in resultados_yolo
            if obj.confidence >= 0.3
        ]
        status = " (Confiança Reduzida)"
    else:
        status = ""

    if not objetos_filtrados:
        return "Nenhum objeto relevante detectado."

    # Contar ocorrências
    contagem = Counter(objetos_filtrados)

    # Montar a string descritiva
    itens = [f"{qtd} {nome}" for nome, qtd in contagem.items()]
    descricao = ", ".join(itens)

    return f"Objetos detectados{status}: {descricao}."

def extrair_dados_yolo(caminho_imagem, modelo):
    # Roda a detecção (o 'stream=True' é mais eficiente em memória)
    resultados_brutos = modelo.predict(source=caminho_imagem, conf=0.3, verbose=False)

    lista_convertida = []

    for r in resultados_brutos:
        # Pega o dicionário de nomes do modelo (ex: {0: 'person', 1: 'bicycle', ...})
        nomes_classes = r.names

        # Itera pelas caixas (boxes) detectadas
        for box in r.boxes:
            id_classe = int(box.cls[0])    # ID numérico
            conf = float(box.conf[0])      # Confiança (0.0 a 1.0)
            nome = nomes_classes[id_classe] # Transforma ID em texto

            # Cria o objeto que sua função 'formatar_objetos_yolo' entende
            lista_convertida.append(ObjectDetected(nome, conf))

    return lista_convertida

In [ ]:
modelo_nano = YOLO('yolov8n.pt')

# No loop de processamento:
dados_brutos = extrair_dados_yolo(caminho_projeto, modelo_nano)
string_para_o_prompt = formatar_objetos_yolo(dados_brutos)

print(string_para_o_prompt)
# Saída esperada: "Objetos detectados: 1 person, 2 cups, 1 laptop."

Objetos detectados: 11 person, 1 potted plant, 1 cup.


## DeepFace

In [ ]:
from deepface import DeepFace
import cv2

def extracao_de_emocoes(caminho_imagem):
    try:
        # Detecção inicial
        # O DeepFace retorna uma lista de resultados
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False # Evita erro se nenhuma face for encontrada
        )

        if not resultados:
            return "Nenhuma expressão facial detectada."

        face = resultados[0]
        emocao_detectada = face['dominant_emotion']
        confianca = face['emotion'][emocao_detectada] / 100 # O DeepFace fornece a pontuação de todas as emoções; pegamos a dominante

        # Verificação de Limite Fixo (0.5 ou 50%)
        if confianca >= 0.5:
            return f"Emoção: {emocao_detectada} (Alta Confiança: {confianca:.2%})"

        # Redução Dinâmica (Se falhar no passo anterior)
        elif confianca >= 0.3:
            return f"Emoção: {emocao_detectada} (Confiança Reduzida: {confianca:.2%} - Tratar como hipótese)"

        # Falha Crítica
        else:
            return "Nenhuma expressão facial detectada com clareza suficiente."

    except Exception as e:
        return f"Erro no processamento de imagem: {str(e)}"

26-04-16 11:24:39 - Directory /root/.deepface has been created
26-04-16 11:24:39 - Directory /root/.deepface/weights has been created


### Múltiplas *faces*

In [ ]:
from deepface import DeepFace
from collections import Counter

def analisar_clima_do_grupo(caminho_imagem):
    try:
        # Usando retinaface para maior precisão em faces pequenas/em movimento
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend='retinaface'
        )

        if not isinstance(resultados, list) or len(resultados) == 0:
            return "Nenhuma expressão facial detectada para análise de grupo."

        num_faces = len(resultados)
        soma_emocoes = {'angry': 0, 'disgust': 0, 'fear': 0, 'happy': 0, 'sad': 0, 'surprise': 0, 'neutral': 0}

        for face in resultados:
            if 'emotion' in face:
                for emocao, percentual in face['emotion'].items():
                    soma_emocoes[emocao] += percentual

        # Média das emoções
        media_emocoes = {e: (v / num_faces) for e, v in soma_emocoes.items()}

        top_emocoes = sorted(media_emocoes.items(), key=lambda item: item[1], reverse=True)[:2]

        emocao_1, val_1 = top_emocoes[0]
        emocao_2, val_2 = top_emocoes[1]

        return f"Expressões Faciais Predominantes: {emocao_1} ({val_1:.1f}%) e {emocao_2} ({val_2:.1f}%) baseado em {num_faces} faces."

    except Exception as e:
        return f"Erro ao processar clima do grupo: {str(e)}"

## Montagem de prompt e comunicação com LLM

In [ ]:
def montagem_de_prompt(caminho_instrucoes, info_yolo, info_deepface, legenda_usuario):
    # 1. Carregamento e Validação das Instruções
    try:
        with open(caminho_instrucoes, 'r') as f:
            instrucoes = f.read()
            # Validação de integridade das tags que definimos
            if "### INSTRUÇÕES" not in instrucoes:
                raise ValueError("Arquivo de instruções inválido: Tag '### INSTRUÇÕES' não encontrada.")
    except FileNotFoundError:
        return "ERRO: O arquivo instructions.md não foi encontrado no caminho especificado."

    # 2. Montagem Estruturada do Prompt
    # Usamos f-strings para injetar as variáveis nas tags correspondentes
    super_prompt = f"{instrucoes}\n\n"
    super_prompt += "### DADOS DA IMAGEM\n"
    super_prompt += f"{info_yolo}\n"
    super_prompt += f"{info_deepface}\n\n"
    super_prompt += f"### LEGENDA ORIGINAL\n\"{legenda_usuario}\"\n\n"
    super_prompt += "### ANÁLISE TÉCNICA\n"

    return super_prompt

In [ ]:
# Comunicação entre meu prompt com a LLM, usando langchain_chain (minha ponte)
def gerar_analise_tecnica(prompt_completo, langchain_chain):
    resposta = langchain_chain.invoke({"question": prompt_completo})
    return resposta

In [ ]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# Configuração do Modelo Local (Ollama)
llm_model = OllamaLLM(model="gemma4:e2b")

# Definição do Template
template = """Responda à pergunta baseada nas instruções técnicas fornecidas.

Pergunta: {question}

Resposta:"""

prompt_template = ChatPromptTemplate.from_template(template)

# Criação da Chain
chain = prompt_template | llm_model

print("✅ Variável 'chain' inicializada com o modelo gemma:2b!")

✅ Variável 'chain' inicializada com o modelo gemma:2b!


## Função de limpeza de gpu

In [ ]:
import torch
import gc

def limpar_memoria_gpu():
    # Coleta de lixo do Python (limpa referências mortas)
    gc.collect()
    # Limpa o cache do PyTorch na GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Conclusão

1. **Automação de Relatórios:** A capacidade de transformar dados brutos de visão computacional em documentos PDF técnicos, prontos para análise acadêmica.

2. **Confiabilidade Multimodal:** A integração entre YOLOv8 e DeepFace provou ser eficaz para capturar a complexidade de cenas reais, identificando discrepâncias entre o que é visto e o que é relatado (legenda).

3. **Eficiência Operacional:** Com a implementação de rotinas de limpeza de memória e orquestração via LangChain, o sistema mantém estabilidade mesmo no processamento de lotes de imagens, garantindo a viabilidade do TCC em hardware acessível.

## Configurando relatório

In [ ]:
from fpdf import FPDF
import os

def gerar_pdf_relatorio(texto_analise, nome_arquivo):
    class PDF(FPDF):
        def header(self):
            self.set_font('Arial', 'B', 15)
            self.set_text_color(50, 50, 50) # Azul Moderno
            self.cell(0, 10, 'RELATÓRIO DE ANÁLISE MULTIMODAL', 0, 1, 'C')
            self.ln(5)

    pdf = PDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # Cores e Formatação
    pdf.set_font('Arial', '', 12)
    pdf.set_text_color(50, 50, 50) # Cinza Escuro

    # Limpeza de caracteres e formatação de tópicos
    texto_limpo = texto_analise.replace('**', '')
    # Troca bullets problemáticos por traços simples
    texto_limpo = texto_limpo.replace('•', '-').replace('●', '-')

    for linha in texto_limpo.split('\n'):
        if ':' in linha and linha.isupper:
            # Destaca chaves em Negrito e Azul
            pdf.set_font('Arial', 'B', 12)
            pdf.set_text_color(33, 150, 243)
            pdf.multi_cell(0, 8, linha.encode('latin-1', 'replace').decode('latin-1'))
            pdf.set_font('Arial', '', 12)
            pdf.set_text_color(50, 50, 50)
        else:
            pdf.multi_cell(0, 8, linha.encode('latin-1', 'replace').decode('latin-1'))

    caminho_pdf = os.path.join(caminho_projeto, f'{nome_arquivo}.pdf')
    pdf.output(caminho_pdf)
    print(f'✅ PDF Customizado gerado em: {caminho_pdf}')
    return caminho_pdf

## Pipeline CI/CD

In [ ]:
import os
import glob

def executar_pipeline(caminho_img, modelo_yolo, modelo_llm, caminho_regras, legenda=None, gerar_pdf=True):
    print(f"[{os.path.basename(caminho_img)}] Iniciando processamento...")

    texto_legenda = legenda if legenda and str(legenda).strip() != "" else "Nenhuma legenda fornecida para esta imagem."

    # Visão Computacional
    dados_yolo = extrair_dados_yolo(caminho_img, modelo_yolo)
    info_yolo = formatar_objetos_yolo(dados_yolo)
    info_deepface = analisar_clima_do_grupo(caminho_img)

    # Orquestração de Prompt
    prompt = montagem_de_prompt(caminho_regras, info_yolo, info_deepface, texto_legenda)

    # Inferência LLM
    print("  -> Analisando contexto multimodal com LLM...")
    analise = gerar_analise_tecnica(prompt, modelo_llm)

    # Saída e Relatório FPDF
    if gerar_pdf:
        print(f"  -> Gerando PDF: Relatorio.pdf")
        gerar_pdf_relatorio(analise, f'Relatorio')

    # 6. LIMPEZA DE MEMÓRIA (AQUI!)
    limpar_memoria_gpu()

    print("  -> Finalizado e Memória Limpa!")
    return analise

# Definindo o processador em lote
def processar_lote_imagens(pasta_imagens, modelo_yolo, modelo_llm, caminho_regras):
    extensoes = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    lista_imagens = []

    for ext in extensoes:
        lista_imagens.extend(glob.glob(os.path.join(pasta_imagens, ext)))

    if not lista_imagens:
        print(f"ATENÇÃO: Nenhuma imagem encontrada na pasta: {pasta_imagens}")
        return

    print(f"Encontradas {len(lista_imagens)} imagens para processar.")

    for caminho_img in lista_imagens:
        nome_arquivo = os.path.basename(caminho_img)
        try:
            executar_pipeline(
                caminho_img=caminho_img,
                legenda=None,
                modelo_yolo=modelo_yolo,
                modelo_llm=modelo_llm,
                caminho_regras=caminho_regras,
                gerar_pdf=True
            )
        except Exception as e:
            print(f"  -> Erro crítico ao processar o arquivo '{nome_arquivo}': {e}")

    print("\nTodos os lotes foram processados com sucesso!")

print("INICIANDO VARREDURA DA PASTA...")
processar_lote_imagens(
    pasta_imagens=caminho_projeto,
    modelo_yolo=model_yolo_nano,
    modelo_llm=chain,
    caminho_regras=caminho_instrucoes
)

INICIANDO VARREDURA DA PASTA...
Encontradas 4 imagens para processar.
[people_runs.jpg] Iniciando processamento...
26-04-16 11:24:48 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5
100%|██████████| 119M/119M [00:01<00:00, 107MB/s]


26-04-16 11:24:55 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 111MB/s]


  -> Analisando contexto multimodal com LLM...
  -> Erro crítico ao processar o arquivo 'people_runs.jpg': model 'gemma4:e2b' not found (status code: 404)
[people_neutral.jpg] Iniciando processamento...
  -> Analisando contexto multimodal com LLM...
  -> Erro crítico ao processar o arquivo 'people_neutral.jpg': model 'gemma4:e2b' not found (status code: 404)
[peoples.jpg] Iniciando processamento...
  -> Analisando contexto multimodal com LLM...
  -> Erro crítico ao processar o arquivo 'peoples.jpg': model 'gemma4:e2b' not found (status code: 404)
[rage people.jpg] Iniciando processamento...
  -> Analisando contexto multimodal com LLM...
  -> Erro crítico ao processar o arquivo 'rage people.jpg': model 'gemma4:e2b' not found (status code: 404)

Todos os lotes foram processados com sucesso!


# Futuras Features

Próximas implementações do TCC, focando em visualização, interatividade e enriquecimento de dados.

### 1. Visualização de Detecções (Bounding Boxes)

Implementação de funções para desenhar quadrados de detecção do YOLO e DeepFace em uma pasta temporária `/content/outputs/` para validação visual do usuário.

In [ ]:
import cv2
import matplotlib.pyplot as plt

def salvar_visualizacao_deteccao(caminho_img, resultados_yolo, resultados_deepface):
    img = cv2.imread(caminho_img)
    os.makedirs('/content/outputs', exist_ok=True)

    # 1. Desenhar YOLO
    for r in resultados_yolo:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2) # Azul para Objetos

    # 2. Desenhar DeepFace
    if isinstance(resultados_deepface, list):
        for face in resultados_deepface:
            region = face['region']
            x, y, w, h = region['x'], region['y'], region['w'], region['h']
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2) # Vermelho para Faces

    nome_saida = os.path.basename(caminho_img)
    caminho_saida = f'/content/outputs/debug_{nome_saida}'
    cv2.imwrite(caminho_saida, img)
    return caminho_saida

### 2. Captura de Imagem via Webcam (JavaScript Bridge)

Como o Colab roda em nuvem, usamos JavaScript para acessar a câmera local do navegador.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def capturar_foto_webcam(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

### 3. Gerenciamento de Legendas via Metadados

Uso de um arquivo `legendas.json` no Google Drive para associar descrições às imagens, fornecendo contexto textual para a análise de dissonância do LLM.

In [ ]:
import json

def carregar_legenda_imagem(nome_imagem, caminho_json='/content/drive/MyDrive/notebookcolab/legendas.json'):
    try:
        with open(caminho_json, 'r', encoding='utf-8') as f:
            base_legendas = json.load(f)
        return base_legendas.get(nome_imagem, "Nenhuma legenda encontrada nos metadados.")
    except Exception as e:
        return "Arquivo de legendas não disponível."